# Wikidata genre tree — exploration

Exploratory notebook over the Bronze `wikidata_genre_tree.parquet` output (see [`../SCHEMA.md`](../SCHEMA.md#bronze)) and the Silver `1_classification`/`2_genre_parents`/`3_hierarchy` outputs (see [`../SCHEMA.md`](../SCHEMA.md#silver)). Reads local Parquet files only — no live SPARQL calls.

**Prerequisite:** run the ingest and silver steps first so the Parquet files exist:

```bash
uv run --package wikidata python -m wikidata.ingest
uv run --package wikidata python -m wikidata.silver
```

In [ ]:
import os
from pathlib import Path

import networkx as nx
import polars as pl
from dotenv import load_dotenv

PIPELINE_ROOT = Path("..").resolve()
load_dotenv(PIPELINE_ROOT / ".env")

bronze_output_dir = Path(os.environ["BRONZE_OUTPUT_DIR"])
if not bronze_output_dir.is_absolute():
    bronze_output_dir = PIPELINE_ROOT / bronze_output_dir

bronze_path = bronze_output_dir / "wikidata_genre_tree.parquet"
df = pl.read_parquet(bronze_path)
df.shape

## Tabular exploration

In [ ]:
df.head(10)

In [ ]:
df.group_by("relation_type").len().sort("len", descending=True)

In [ ]:
roots = df.filter(pl.col("parent_id").is_null())
print(f"{roots.height} root items")
roots.select("item_id", "item_label").head(10)

In [ ]:
multi_parent = (
    df.filter(pl.col("parent_id").is_not_null())
    .group_by("item_id", "item_label")
    .agg(pl.col("parent_id").n_unique().alias("n_parents"))
    .filter(pl.col("n_parents") > 1)
    .sort("n_parents", descending=True)
)
print(f"{multi_parent.height} items with more than one parent")
multi_parent.head(10)

## Graph visualization

The full tree (~9,700 edges) is too dense to render usefully in one plot, so build the full graph for structural stats, then plot just the neighborhood around a single genre.

In [ ]:
G = nx.DiGraph()
for row in df.iter_rows(named=True):
    G.add_node(row["item_id"], label=row["item_label"])
    if row["parent_id"] is not None:
        G.add_edge(row["item_id"], row["parent_id"], relation=row["relation_type"])

print(f"{G.number_of_nodes()} nodes, {G.number_of_edges()} edges")
print(f"{nx.number_weakly_connected_components(G)} weakly connected components")

In [ ]:
import matplotlib.pyplot as plt

ITEM_ID = "Q11399"  # rock music — change this to explore a different genre
RADIUS = 1  # networkx's spring_layout needs scipy for subgraphs of 500+ nodes; radius=1 stays well under that

undirected = G.to_undirected()
subgraph = nx.ego_graph(undirected, ITEM_ID, radius=RADIUS)


def edge_relation(u: str, v: str) -> str | None:
    return G.get_edge_data(u, v, {}).get("relation") or G.get_edge_data(v, u, {}).get("relation")


edge_colors = ["tab:blue" if edge_relation(u, v) == "P279" else "tab:orange" for u, v in subgraph.edges()]
labels = {n: G.nodes[n].get("label", n) for n in subgraph.nodes()}

plt.figure(figsize=(12, 10))
pos = nx.spring_layout(subgraph, seed=42)
nx.draw_networkx_nodes(subgraph, pos, node_size=300, node_color="lightgray")
nx.draw_networkx_edges(subgraph, pos, edge_color=edge_colors, alpha=0.6)
nx.draw_networkx_labels(subgraph, pos, labels=labels, font_size=8)
plt.title(f"Genre tree neighborhood: {labels[ITEM_ID]} (radius={RADIUS})\nblue = P279, orange = P361")
plt.axis("off")
plt.show()

## Silver exploration

`2_genre_parents.parquet` carries both Silver columns — `is_genre`/`exclusion_reason` from `1_classification`, plus `parent_is_genre` from `2_genre_parents` — on top of the unchanged Bronze edge list.

In [ ]:
silver_output_dir = Path(os.environ["SILVER_OUTPUT_DIR"])
if not silver_output_dir.is_absolute():
    silver_output_dir = PIPELINE_ROOT / silver_output_dir

silver_df = pl.read_parquet(silver_output_dir / "2_genre_parents.parquet")
silver_df.shape

In [ ]:
silver_df.group_by("is_genre", "exclusion_reason").len().sort("len", descending=True)

In [ ]:
excluded = silver_df.filter(~pl.col("is_genre")).select("item_id", "item_label", "exclusion_reason").unique()
print(f"{excluded.height} distinct items excluded")
excluded.head(10)

In [ ]:
silver_df.group_by("parent_is_genre").len().sort("len", descending=True)

In [ ]:
non_genre_parents = (
    silver_df.filter(~pl.col("parent_is_genre"))
    .select("item_id", "item_label", "parent_id", "parent_label", "relation_type")
    .unique()
)
print(f"{non_genre_parents.height} edges into a non-genre parent")
non_genre_parents.head(10)

### 3_hierarchy — pruned, single-parent-per-item

The first Silver step that drops rows: filters to genre-only edges, then collapses any item with more than one surviving genre parent down to the one with the lowest numeric QID — a **provisional heuristic** (see [`../SCHEMA.md#3_hierarchy`](../SCHEMA.md#3_hierarchy)), not a considered rule.

In [ ]:
hierarchy_df = pl.read_parquet(silver_output_dir / "3_hierarchy.parquet")
hierarchy_df.shape

In [ ]:
genre_items = silver_df.filter(pl.col("is_genre")).select("item_id").unique()
surviving_items = hierarchy_df.select("item_id").unique()
vanished = genre_items.join(surviving_items, on="item_id", how="anti")
print(
    f"{vanished.height} of {genre_items.height} genre items have zero surviving rows (every parent edge was non-genre)"
)
vanished.head(10)

In [ ]:
# Sample of items with more than one genre-only parent candidate, and which one the lowest-QID
# heuristic kept (chosen_parent_id) vs. the alternatives it discarded.
multi_parent_candidates = silver_df.filter(pl.col("is_genre") & pl.col("parent_is_genre")).select(
    "item_id", "item_label", "parent_id", "parent_label"
)
multi_parent_items = multi_parent_candidates.group_by("item_id").len().filter(pl.col("len") > 1).select("item_id")
print(f"{multi_parent_items.height} items have more than one genre parent")

sample = (
    multi_parent_candidates.join(multi_parent_items, on="item_id")
    .join(hierarchy_df.select("item_id", pl.col("parent_id").alias("chosen_parent_id")), on="item_id")
    .sort("item_id")
)
sample.head(10)